# Marathos Country EDA from bronze layer to silver layer cleaning

In [0]:
from pyspark.sql.functions import col, trim, upper, count, when

In [0]:
# Load Bronze tables for Silver cleaning trial
country_codes_bronze_df = spark.sql("""
    SELECT *
    FROM marathos_catalog.bronze.raw_country_codes
""")

display(country_codes_bronze_df)

#### check row count and schema

In [0]:
# Check country Bronze row count and schema
print(f"Country rows: {country_codes_bronze_df.count()}")

country_codes_bronze_df.printSchema()

#### clean values

In [0]:
# Standardize country mapping columns
country_codes_silver_df = (
    country_codes_bronze_df
    .withColumn("athlete_country_code", upper(trim(col("athlete_country_code"))))
    .withColumn("country_code_iso3", upper(trim(col("country_code_iso3"))))
    .withColumn("country_name", trim(col("country_name")))
)

display(country_codes_silver_df)

#### NULLS

In [0]:
# Check nulls in country mapping columns
country_nulls_df = country_codes_silver_df.select(
    count(when(col("athlete_country_code").isNull(), True)).alias("null_athlete_country_code"),
    count(when(col("country_code_iso3").isNull(), True)).alias("null_country_code_iso3"),
    count(when(col("country_name").isNull(), True)).alias("null_country_name")
)

display(country_nulls_df)

#### Duplicates

In [0]:
# Check if athlete_country_code is unique
duplicate_country_codes_df = (
    country_codes_silver_df
    .groupBy("athlete_country_code")
    .agg(count("*").alias("row_count"))
    .filter(col("row_count") > 1)
)

display(duplicate_country_codes_df)

In [0]:
# Display final cleaned country mapping
display(
    country_codes_silver_df
    .orderBy("athlete_country_code")
)

#### Results:
- dataset contains 205 rows.
- dataset has clean country code fields and is ready to be joined with the marathon Silver OBT. 
- The athlete_country_code column will match the athlete_country column from the marathon dataset.